# DistilBERT Fine-tuning for Incident Classification

This notebook fine-tunes `distilbert-base-uncased` on the 3-class incident dataset.
After fine-tuning, the backbone's CLS embeddings will have meaningful separation between
authentication failures, network issues, and deployment issues — enabling DBSCAN to cluster
correctly based on semantic similarity alone, not just the time window.

**Runtime:** ~3 min on T4 GPU, ~15 min on CPU.

**Output:** `distilbert-finetuned.zip` — download and extract to `models/distilbert-finetuned/` in your project.

## Step 1 — Install dependencies

Run the cell below, then **Runtime → Restart session** from the menu. After restart, skip this cell and continue from Step 2.

> **GPU CUDA errors?** Switch to CPU: `Runtime → Change runtime type → None`. No CUDA = no mismatch. Fine-tuning 450 samples takes ~15 min on CPU.

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
!pip install transformers accelerate datasets

## Step 2 — Upload `synthetic_incidents.csv`

Run this cell and select the file from `data/raw/synthetic_incidents.csv` in your project.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select synthetic_incidents.csv

## Step 3 — Imports and data loading

In [ ]:
import sys
import types

# Colab T4 sometimes has a CUDA version mismatch between torch and torchvision.
# This notebook doesn't use torchvision, so intercept its import before
# transformers triggers it and raises a RuntimeError.
try:
    import torchvision  # noqa: F401
except RuntimeError:
    _stub = types.ModuleType("torchvision")
    _stub.__version__ = "0.0.0"
    sys.modules["torchvision"] = _stub

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity
from torch.utils.data import Dataset
from transformers import (
    AutoModel,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
df = pd.read_csv("synthetic_incidents.csv")
print(f"Loaded {len(df)} rows")
print(df["label"].value_counts())

# Singletons (label='none') are excluded from fine-tuning —
# the model only needs to learn the 3 real incident classes.
df_train = df[df["label"] != "none"].reset_index(drop=True)
print(f"\nTraining on {len(df_train)} rows (excluding singletons)")

LABEL_CLASSES = sorted(df_train["label"].unique().tolist())
LABEL2ID = {l: i for i, l in enumerate(LABEL_CLASSES)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}
print(f"Classes: {LABEL_CLASSES}")

df_train["label_id"] = df_train["label"].map(LABEL2ID)

## Step 4 — Dataset class and tokenizer

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class IncidentDataset(Dataset):
    def __init__(self, texts: list[str], labels: list[int], max_length: int = 128):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt",
        )
        self.labels = labels

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> dict:
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_train["text"].tolist(),
    df_train["label_id"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df_train["label_id"].tolist(),
)

train_dataset = IncidentDataset(train_texts, train_labels)
val_dataset = IncidentDataset(val_texts, val_labels)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

## Step 5 — Fine-tune DistilBERT

A classification head is added on top of the DistilBERT backbone and trained end-to-end.
The gradient from the 3-class loss flows back through all 66M parameters, reorganising the
CLS embedding space so that the 3 incident types land in clearly separated regions.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": float(accuracy_score(labels, preds))}


training_args = TrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=20,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
# Final evaluation on validation set
results = trainer.evaluate()
print(f"\nVal accuracy: {results['eval_accuracy']:.4f}")

# Per-class report
preds_output = trainer.predict(val_dataset)
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = preds_output.label_ids
print("\n" + classification_report(y_true, y_pred, target_names=LABEL_CLASSES))

## Step 6 — Compare embeddings: base vs fine-tuned

Extract CLS embeddings from both the base and fine-tuned backbone, then measure
within-group and cross-group cosine similarity. The gap between the two should be
much larger after fine-tuning.

In [ ]:
def extract_cls(backbone, texts: list[str], batch_size: int = 32) -> np.ndarray:
    backbone.eval()
    device = next(backbone.parameters()).device  # detect where model lives (cpu or cuda)
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        enc = tokenizer(
            batch, truncation=True, padding=True, max_length=128, return_tensors="pt"
        )
        enc = {k: v.to(device) for k, v in enc.items()}  # move inputs to same device as model
        with torch.no_grad():
            out = backbone(**enc)
        all_embs.append(out.last_hidden_state[:, 0, :].cpu().numpy())
    return np.vstack(all_embs)


def within_group_sim(embs: np.ndarray, group_ids: np.ndarray) -> float:
    scores = []
    for gid in np.unique(group_ids):
        grp = embs[group_ids == gid]
        if grp.shape[0] < 2:
            continue
        sims = cosine_similarity(grp)
        n = sims.shape[0]
        upper = sims[np.triu_indices(n, k=1)]
        scores.append(float(upper.mean()))
    return float(np.mean(scores))


def cross_group_sim(embs: np.ndarray, labels: np.ndarray, n_pairs: int = 500) -> float:
    rng = np.random.default_rng(42)
    unique_labels = np.unique(labels)
    scores = []
    for _ in range(n_pairs):
        l1, l2 = rng.choice(unique_labels, size=2, replace=False)
        i1 = rng.choice(np.where(labels == l1)[0])
        i2 = rng.choice(np.where(labels == l2)[0])
        sim = float(
            cosine_similarity(embs[i1].reshape(1, -1), embs[i2].reshape(1, -1))[0, 0]
        )
        scores.append(sim)
    return float(np.mean(scores))


# Use only real incident rows (exclude singletons)
texts_real = df_train["text"].tolist()
group_ids_real = df_train["incident_group_id"].to_numpy()
labels_real = df_train["label"].to_numpy()

print("Extracting base DistilBERT embeddings...")
base_backbone = AutoModel.from_pretrained(MODEL_NAME)
base_embs = extract_cls(base_backbone, texts_real)

print("Extracting fine-tuned DistilBERT embeddings...")
ft_embs = extract_cls(model.distilbert, texts_real)

w_base = within_group_sim(base_embs, group_ids_real)
w_ft   = within_group_sim(ft_embs,   group_ids_real)
c_base = cross_group_sim(base_embs, labels_real)
c_ft   = cross_group_sim(ft_embs,   labels_real)

print("\n" + "=" * 62)
print(f"{'Metric':<38} {'Base':>10} {'Fine-tuned':>12}")
print("-" * 62)
print(f"{'Within-group cosine similarity':<38} {w_base:>10.4f} {w_ft:>12.4f}")
print(f"{'Cross-group cosine similarity':<38} {c_base:>10.4f} {c_ft:>12.4f}")
print(f"{'Discrimination gap (within - cross)':<38} {w_base-c_base:>10.4f} {w_ft-c_ft:>12.4f}")
print("=" * 62)
gap_improvement = (w_ft - c_ft) / max(w_base - c_base, 1e-6)
print(f"\nGap is {gap_improvement:.1f}x larger after fine-tuning.")
print("A larger gap means DBSCAN can distinguish incident types")
print("by embedding similarity alone, not just the time window.")

## Step 7 — Save backbone and download

We save only the DistilBERT backbone weights (not the classification head).
The backbone is what `embed.py` loads to extract CLS embeddings.

In [ ]:
import shutil

SAVE_DIR = "./distilbert-finetuned"

# Save the backbone (DistilBERT without the classification head)
model.distilbert.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"Saved backbone to {SAVE_DIR}")

# Zip and download
shutil.make_archive("distilbert-finetuned", "zip", ".", "distilbert-finetuned")
files.download("distilbert-finetuned.zip")

print("\nNext steps:")
print("1. Extract distilbert-finetuned.zip")
print("2. Place the 'distilbert-finetuned' folder at models/distilbert-finetuned/ in your project")
print("3. Run: python src/features/embed.py --model finetuned")
print("4. Open the dashboard and toggle the model in the sidebar")